[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/longas14/Fisica_Computacional_1/blob/main/notebooks/ecuaciones_nolineales.ipynb)

---
# Física Computacional 1
Robinson Longas Bedoya  
*Facultad de Ciencias Exactas y Aplicadas - Institución Universitaria ITM*  
Contacto: [robinsonlongas@itm.edu.co](mailto:robinsonlongas@itm.edu.co) | [robinsonlongas3586@correo.itm.edu.co](mailto:robinsonlongas3586@correo.itm.edu.co)

---

### Temas principales:
* **Solución de ecuaciones no lineales y trascendentes**
* **Raíces de polinomios**

#### Bibliografía recomendada
* *Computational Physics*, Mark Newman.
* *Programming for computations - Python*, Svein Linge and Petter Langtangen.
* *Computational Physics with Python*, Eric Ayars.
* *Computational mathematics: An introduction to Numerical Analysis and Scientific Computing with Python*, Dimitrios Mitsotakis.
* *Computational physics: problem solving with Python*, Rubin H. Landau, Manuel J. Páez and Cristian C. Bordeianu.

## Solución de ecuaciones no lineales y trascendentes

Suponga que se lanza un cuerpo con una velocidad inicial $\vec{v}_0$ formando un ángulo $\theta$ con la horizontal. En física básica se demuestra que las ecuaciones cinemáticas para este sistema son:

$$
\begin{aligned}
x(t) &= x_0 + v_0 \cos\theta \, t, \\
y(t) &= y_0 + v_0 \sin\theta \, t - \frac{1}{2} g t^2.
\end{aligned}
$$

El tiempo $T$ en el cual el proyectil impacta contra el suelo se obtiene al resolver $y(T) = 0$:

$$
T = \frac{v_0 \sin\theta + \sqrt{(v_0 \sin\theta)^2 + 2 g y_0}}{g}.
$$

Sustituyendo este tiempo en la ecuación horizontal se obtiene el alcance horizontal del proyectil:

$$
x(\theta) = x_0 + \frac{v_0 \cos\theta}{g} \left[ v_0 \sin\theta + \sqrt{(v_0 \sin\theta)^2 + 2 g y_0} \right].
$$

Esta expresión es una ecuación no lineal en $\theta$. Si conocemos el ángulo de salida podemos calcular directamente el alcance; sin embargo, en el laboratorio habitualmente medimos el alcance experimental $x$ y deseamos deducir el ángulo de disparo $\theta$. Esto requiere resolver numéricamente la ecuación $f(\theta) = x(\theta) - x_{\text{deseado}} = 0$.

In [ ]:
import scipy as sp
import numpy as np
import matplotlib.pyplot as plt

def x_max(theta, x0=0., y0=0., v0=3.5, g=9.8):
    """Calcula el desplazamiento horizontal máximo en función del ángulo theta en grados."""
    theta_rad = np.radians(theta)
    sintheta = np.sin(theta_rad)
    costheta = np.cos(theta_rad)
    prefactor = (v0 * costheta) / g
    return x0 + prefactor * (v0 * sintheta + np.sqrt(v0**2 * sintheta**2 + 2 * g * y0))

def y_max(theta, y0=0., v0=3.5, g=9.8):
    """Calcula la altura máxima alcanzada por el proyectil."""
    theta_rad = np.radians(theta)
    sintheta = np.sin(theta_rad)
    return y0 + (v0**2 * sintheta**2) / (2.0 * g)

def f_x(theta, xmaximo, x0, y0, v0, g):
    """Función objetivo: x_max(theta, ...) - xmaximo = 0"""
    return x_max(theta, x0, y0, v0, g) - xmaximo

def f_y(theta, ymaxima, y0, v0, g):
    """Función objetivo para altura máxima: y_max(theta, ...) - ymaxima = 0"""
    return y_max(theta, y0, v0, g) - ymaxima

In [ ]:
# Parámetros físicos del sistema
y0 = 1.5       # Altura inicial (m)
x0 = 0.0       # Posición horizontal inicial (m)
v0 = 35.0      # Velocidad inicial (m/s)
g = 9.8        # Aceleración de la gravedad (m/s^2)
xmaximo = 89.0 # Alcance deseado (m)

angulos = np.linspace(0, 90, 300)
alcancesx = x_max(angulos, x0, y0, v0, g)

plt.figure(figsize=(8, 5))
plt.plot(angulos, alcancesx, label=r'$x_{\mathrm{max}}(\theta)$', color='blue', lw=2)
plt.axhline(xmaximo, color='red', linestyle='--', label=rf'$x_{{\mathrm{{obj}}}} = {xmaximo}\ \mathrm{{m}}$')
plt.title(f'Alcance Horizontal vs Ángulo ($v_0 = {v0}$ m/s, $y_0 = {y0}$ m)', fontsize=12)
plt.xlabel(r'$\theta$ (grados)', fontsize=12)
plt.ylabel('$x$ (m)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

try:
    sol1 = sp.optimize.root_scalar(f_x, args=(xmaximo, x0, y0, v0, g), bracket=[0, 45], method='brentq')
    sol2 = sp.optimize.root_scalar(f_x, args=(xmaximo, x0, y0, v0, g), bracket=[45, 90], method='brentq')

    res1 = f_x(sol1.root, xmaximo, x0, y0, v0, g)
    res2 = f_x(sol2.root, xmaximo, x0, y0, v0, g)

    print(f"\n--- Ángulos calculados para un alcance de {xmaximo} metros ---")
    print(f"Ángulo 1 : {sol1.root:.4f}° | Residuo: {abs(res1):.2e} m | Iteraciones: {sol1.iterations}")
    print(f"Ángulo 2 : {sol2.root:.4f}° | Residuo: {abs(res2):.2e} m | Iteraciones: {sol2.iterations}")

except ValueError:
    alcance_max_posible = np.max(alcancesx)
    print(f"\n[Error] El alcance deseado de {xmaximo} m es físicamente inalcanzable.")
    print(f"El alcance máximo posible con v0={v0} m/s y y0={y0} m es de {alcance_max_posible:.2f} m.")

### Ejemplo: $x = -\ln(x)$ y la Función $W$ de Lambert

Encontremos numéricamente la solución de la ecuación:

$$x = -\ln(x) \iff f(x) = x + \ln(x) = 0$$

> **La función $W$ de Lambert:**  
> Se define como la función inversa de $f(w) = w e^w$, tal que:
> 
> $$z = W(z) e^{W(z)}$$
> 
> Manipulando nuestra ecuación algebraica:
> 
> $$x + \ln x = 0 \implies x = -\ln x \implies e^x = e^{-\ln x} = \frac{1}{x} \implies x e^x = 1$$
> 
> Comparando con la definición, la solución exacta a la ecuación $x e^x = 1$ es $x = W(1)$.

In [ ]:
import scipy as sp
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return x + np.log(x)

def df(x):
    return 1.0 + 1.0 / x

In [ ]:
x = np.linspace(0.01, 5, 200)

plt.figure(figsize=(8, 5))
plt.plot(x, x, color='blue', lw=2, label=r'$y = x$')
plt.plot(x, -np.log(x), color='red', linestyle='--', lw=2, label=r'$y = -\ln(x)$')
plt.axhline(0, color='black', lw=0.6, alpha=0.5)
plt.ylabel(r'$f(x)$', fontsize=14)
plt.xlabel(r'$x$', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

# Ejecución de los diferentes algoritmos de búsqueda de raíces
sol_brent   = sp.optimize.root_scalar(f, bracket=[0.1, 1.0], method='brentq')
sol_bisect  = sp.optimize.root_scalar(f, bracket=[0.1, 1.0], method='bisect', xtol=1e-8)
sol_secante = sp.optimize.root_scalar(f, x0=1.0, method='secant')
sol_newton  = sp.optimize.root_scalar(f, x0=1.0, fprime=df, method='newton')
sol_lambert = float(np.real(sp.special.lambertw(1)))

metodos = [
    ("Brent (brentq)", sol_brent.root, sol_brent.iterations, sol_brent.function_calls),
    ("Bisección", sol_bisect.root, sol_bisect.iterations, sol_bisect.function_calls),
    ("Secante", sol_secante.root, sol_secante.iterations, sol_secante.function_calls),
    ("Newton", sol_newton.root, sol_newton.iterations, sol_newton.function_calls),
    ("Lambert W (analítico)", sol_lambert, 1, 1)
]

print(f"{'Método':<22} | {'Raíz Hallada':<14} | {'Residuo |f(x)|':<14} | {'Iteraciones':<12} | {'Llamadas f(x)':<14}")
print("-" * 84)
for nombre, raiz, iters, nfev in metodos:
    residuo = abs(f(raiz))
    print(f"{nombre:<22} | {raiz:<14.8f} | {residuo:<14.2e} | {iters:<12} | {nfev:<14}")

> ### Resumen: Argumentos y Métodos de `scipy.optimize.root_scalar`
> Documentación oficial: [`scipy.optimize.root_scalar`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html)
>
> ```python
> sol = sp.optimize.root_scalar(f, args=(), method=None, bracket=None, fprime=None,
>                               fprime2=None, x0=None, x1=None, xtol=None, rtol=None,
>                               maxiter=None, options=None)
> ```
>
> #### 1. Parámetros Generales
> * **`f`**: Función escalar a resolver $f(x) = 0$.
> * **`args`**: Tupla con constantes extra para la función `f(x, a, b, ...)`.
> * **`xtol` / `rtol`**: Tolerancia absoluta y relativa en la variable independiente $x$.
> * **`maxiter`**: Número máximo de iteraciones permitidas.
> * **`options`**: Diccionario con opciones avanzadas de depuración (ej. `{'disp': True}`).
>
> #### 2. Métodos Principales
> * **Brent (`method='brentq'`)**: Método híbrido cerrado recomendado por defecto. Combina bisección, secante e interpolación inversa. Requiere obligatoriamente un cambio de signo en `bracket=[a, b]`.
> * **Bisección (`method='bisect'`)**: Método cerrado robusto pero de convergencia lineal (lenta). Requiere cambio de signo en `bracket=[a, b]`.
> * **Secante (`method='secant'`)**: Método abierto superlineal ($\approx 1.618$). No requiere derivadas ni cambio de signo inicial; requiere semilla `x0` (y opcionalmente `x1`).
> * **Newton-Raphson (`method='newton'`)**: Método abierto cuadrático. Requiere semilla `x0` y la primera derivada en `fprime`.
>
> #### 3. Atributos del Objeto `sol`
> * `sol.root`: Valor numérico de la raíz.
> * `sol.converged`: Booleano (`True`/`False`) de convergencia.
> * `sol.iterations`: Cantidad de iteraciones efectuadas.
> * `sol.function_calls`: Cantidad de veces que se evaluó la función $f(x)$.

## Raíces de Polinomios

El cálculo analítico de raíces para polinomios de grado $n > 3$ es en general inviable. Consideremos por ejemplo el polinomio:

$$p(x) = 3x^7 - 4x^5 + 6x^2 - x + 3$$

NumPy ofrece dos enfoques:
1. La función clásica basada en coeficientes descendentes: `np.roots`.
2. La API moderna orientada a objetos: `numpy.polynomial.Polynomial` ([Documentación oficial](https://numpy.org/doc/stable/reference/routines.polynomials.html)), la cual ofrece mayor estabilidad numérica calculando los autovalores de la matriz compañera escalada.

In [ ]:
"""
Programa: raices_polinomios.py
Cálculo de raíces y graficación utilizando la API moderna numpy.polynomial.Polynomial
"""

import numpy as np
import matplotlib.pyplot as plt

# Polinomio: 3x^7 - 4x^5 + 0x^4 + 0x^3 + 6x^2 - x + 3 = 0
# Coeficientes en orden ASCENDENTE de potencias (a_0, a_1, ..., a_n)
coeff = [3, -1, 6, 0, 0, -4, 0, 3]

p = np.polynomial.Polynomial(coeff)
raices_p = p.roots()

print("Polinomio instanciado:")
print(p)

print("\nRaíces calculadas:")
for i, r in enumerate(raices_p):
    valorevaluado = p(r)
    print(f"  x_{i+1} = {r:.4f}  |  Residuo |P(x)|: {abs(valorevaluado):.2e}")

# Gráfica del polinomio
x = np.linspace(-1.6, 1.3, 400)
y = p(x)

plt.figure(figsize=(9, 6))
plt.plot(x, y, color='red', lw=2, label=r'$p(x) = 3x^7 - 4x^5 + 6x^2 - x + 3$')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)

# Filtrar y marcar las raíces reales sobre la curva
raices_reales = raices_p[np.isreal(raices_p)].real
if len(raices_reales) > 0:
    plt.plot(raices_reales, p(raices_reales), 'ko', markersize=7, label='Raíces reales')

plt.xlabel(r'$x$', fontsize=14)
plt.ylabel(r'$p(x)$', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(fontsize=12)
plt.tick_params(axis='both', which='both', labelsize=12)
plt.tight_layout()
plt.show()

> ### Resumen: `numpy.polynomial.Polynomial`
> Documentación: [`numpy.polynomial.Polynomial`](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.Polynomial.html)
>
> ```python
> p = np.polynomial.Polynomial(coeff, domain=None, window=None, symbol='x')
> ```
>
> 1. **Orden de coeficientes:** Requiere orden **ascendente** ($a_0 + a_1 x + a_2 x^2 + \dots$). Los coeficientes de potencias ausentes deben definirse explícitamente con cero.
> 2. **Símbolo algebraico:** El parámetro `symbol='t'` permite cambiar la variable simbólica al imprimir.
> 3. **Cálculo de raíces (`p.roots()`):** Obtiene todas las raíces complejas calculando los autovalores de la matriz compañera.
> 4. **Evaluación directa (`p(x)`):** Permite evaluar escalares o arreglos vectorizados directamente llamando al objeto como una función.

## Problemas Propuestos

---

### Problema 1: Ecuaciones no lineales elementales
Resuelva las siguientes ecuaciones combinando inspección gráfica y las rutinas de `scipy.optimize`:
1. $x = 2 - e^{-x}$
2. $x^3 + 2x = e^{-x}$
3. $x = \cos(x)$

---

### Problema 2: Ley de Radiación de Planck y Desplazamiento de Wien
La ley de Planck describe la radiancia espectral de un cuerpo negro a temperatura $T$:

$$I(\lambda) = \frac{2\pi h c^2 \lambda^{-5}}{e^{hc/\lambda k_B T} - 1}$$

a) Deduzca que el máximo de emisión satisface la ecuación no lineal:

$$5e^{-x} + x - 5 = 0, \quad \text{donde} \quad x = \frac{hc}{\lambda k_B T}.$$

Demuestre que esto equivale a la **Ley de Desplazamiento de Wien**: $\lambda_{\text{max}} = \frac{b}{T}$, con $b = \frac{hc}{k_B x}$.

b) Implemente un programa que resuelva esta ecuación con tolerancia $\epsilon = 10^{-6}$ usando el método de bisección y estime el valor de la constante $b$.

c) Con el pico del espectro solar en $\lambda \approx 502\text{ nm}$, estime la temperatura superficial del Sol por pirometría óptica.

---

### Problema 3: Modos Normales de una Viga Empotrada-Libre
La vibración elástica de una viga conduce a la ecuación trascendente:

$$\cosh(\beta) \cos(\beta) = -1, \quad \text{con} \quad \beta^4 = \omega^2 \frac{\rho A}{E I}.$$

Para una barra de acero con $b = 25\text{ mm}$, $h = 8\text{ mm}$, $\rho = 7850\text{ kg/m}^3$, $E = 2 \times 10^{11}\text{ Pa}$ e $I = \frac{bh^3}{12}$:

a) Grafique la versión amortiguada $g(\beta) = e^{-\beta} [\cosh(\beta)\cos(\beta) + 1] = 0$ para inspeccionar las raíces.  
b) A partir de las tres primeras raíces $\beta_1, \beta_2, \beta_3$, calcule las tres primeras frecuencias naturales $\omega_1, \omega_2, \omega_3$ en $\text{rad/s}$ (y en $\text{Hz}$).

---

### Problema 4: Polinomio de Legendre Modificado
Considere el polinomio:

$$P(x) = 924x^6 - 2772x^5 + 3150x^4 - 1680x^3 + 420x^2 - 42x + 1$$

a) Grafique $P(x)$ en $x \in [0, 1]$ para aproximar visualmente las 6 raíces reales.  
b) Calcule numéricamente todas las raíces usando `numpy.polynomial.Polynomial` y muestre los residuos correspondientes $\vert{}P(x_i)\vert{}$.

---

### Problema 5: Punto de Lagrange $L_1$ (Sistema Tierra-Luna)
Para un satélite colocado entre la Tierra y la Luna a una distancia $r$ del centro terrestre, el equilibrio dinámico orbital impone:

$$\frac{GM}{r^2} - \frac{Gm}{(R - r)^2} = \omega^2 r$$

Multiplicando por $r^2(R-r)^2$ se obtiene el polinomio de quinto grado:

$$\omega^2 r^5 - 2\omega^2 R r^4 + \omega^2 R^2 r^3 + G(m - M) r^2 + 2 G M R r - G M R^2 = 0$$

*Datos del sistema:*
* $G = 6.674 \times 10^{-11}\text{ m}^3\text{kg}^{-1}\text{s}^{-2}$
* $M = 5.974 \times 10^{24}\text{ kg}$ (Masa Tierra)
* $m = 7.348 \times 10^{22}\text{ kg}$ (Masa Luna)
* $R = 3.844 \times 10^8\text{ m}$ (Distancia Tierra-Luna)
* $\omega = 2.662 \times 10^{-6}\text{ s}^{-1}$

Escriba un script que determine las raíces con `Polynomial.roots()`, identifique la raíz físicamente admisible ($0 < r < R$) y calcule a qué distancia de la superficie terrestre se encuentra $L_1$.